# 01 — Setup & Configuration

Browser setup, config loading, and common utilities.

In [14]:
import os, sys, time, glob, shutil, yaml
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    ElementClickInterceptedException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager

In [15]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CONFIG_PATH  = os.path.join(PROJECT_ROOT, "config", "config.yaml")

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

DATA_RAW       = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
TEMP_DOWNLOAD  = os.path.join(PROJECT_ROOT, "data", "_temp_downloads")
os.makedirs(TEMP_DOWNLOAD, exist_ok=True)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.trading_calendar import resolve_session_date, describe_session, now_ist

# NSE session the data belongs to (IST + NSE holidays), not the computer's date.
TODAY = resolve_session_date()

print(f"Project root: {PROJECT_ROOT}")
print(describe_session())

Project root: C:\Users\sarda\Desktop\nse-options-scraper
Today: 2026-03-21


In [16]:
def get_driver():
    chrome_options = Options()
    prefs = {
        "download.default_directory": TEMP_DOWNLOAD,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    if CONFIG["scraping"].get("headless", False):
        chrome_options.add_argument("--headless=new")

    # Let Selenium handle chromedriver automatically (no webdriver-manager needed)
    driver = webdriver.Chrome(options=chrome_options)

    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    driver.set_page_load_timeout(CONFIG["scraping"]["page_load_timeout"])
    return driver

In [17]:
NSE_BASE = "https://www.nseindia.com"
NSE_OPTION_CHAIN = f"{NSE_BASE}/option-chain"

def init_nse_session(driver):
    print("Visiting NSE homepage for session cookies...")
    driver.get(NSE_BASE)
    time.sleep(4)
    print("Navigating to Option Chain page...")
    driver.get(NSE_OPTION_CHAIN)
    time.sleep(5)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "equity_optionchain_select"))
    )
    print("Option chain page loaded.")

In [18]:
def wait_for_download(before_files, timeout=30):
    end_time = time.time() + timeout
    while time.time() < end_time:
        current = set(os.listdir(TEMP_DOWNLOAD))
        new_files = current - before_files
        new_files = {f for f in new_files if not f.endswith(".crdownload") and not f.endswith(".tmp")}
        if new_files:
            path = os.path.join(TEMP_DOWNLOAD, new_files.pop())
            time.sleep(0.5)
            return path
        time.sleep(0.5)
    raise TimeoutError(f"No new file in {TEMP_DOWNLOAD} after {timeout}s")


def clear_temp_downloads():
    for f in os.listdir(TEMP_DOWNLOAD):
        fp = os.path.join(TEMP_DOWNLOAD, f)
        if os.path.isfile(fp):
            os.remove(fp)


def safe_click(driver, element):
    try:
        element.click()
    except ElementClickInterceptedException:
        driver.execute_script("arguments[0].click();", element)


def slow_delay():
    time.sleep(CONFIG["scraping"]["delay_between_requests"])


def close_chart_modal(driver):
    try:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
        time.sleep(1)
    except Exception:
        pass
    for sel in [
        '//button[contains(@class, "close")]',
        '//span[contains(@class, "close")]',
        '//div[contains(@class, "modal")]//button',
    ]:
        try:
            btn = driver.find_element(By.XPATH, sel)
            if btn.is_displayed():
                safe_click(driver, btn)
                time.sleep(1)
                return
        except (NoSuchElementException, StaleElementReferenceException):
            continue

In [ ]:
# ── Watchlist & symbol helpers ────────────────────────────────
INDEX_SYMBOLS = {"NIFTY", "BANKNIFTY", "FINNIFTY", "MIDCPNIFTY", "NIFTYNXT50"}
WEEKLY_INDEX_SYMBOLS = {"NIFTY"}
MONTHLY_INDEX_SYMBOLS = {"BANKNIFTY", "FINNIFTY", "MIDCPNIFTY"}

SYMBOL_ALIASES = {
    "FINNIFTY1!": "FINNIFTY",
    "MIDCPNIFTY1!": "MIDCPNIFTY",
    "BAJAJ_AUTO": "BAJAJ-AUTO",
}

XPATH_INDEX_DROPDOWN  = '//select[@id="equity_optionchain_select"]'
XPATH_EQUITY_DROPDOWN = '//select[@id="select_symbol"]'
XPATH_EXPIRY_DROPDOWN = '//select[@id="expirySelect"]'
# NSE renamed inner span id: download_csv -> download_csv1 (Jul 2026)
XPATH_DOWNLOAD_CSV    = '//a[@id="downloadOCTable"]'
XPATH_CHART_MENU      = '//div[@id="modal_oc_graph"]//*[contains(@class,"highcharts-contextbutton")]'


def normalize_symbol(symbol):
    s = symbol.upper().strip()
    return SYMBOL_ALIASES.get(s, s)


def get_symbol_profile(symbol):
    symbol = normalize_symbol(symbol)
    if symbol in WEEKLY_INDEX_SYMBOLS:
        return {"asset_class": "index", "index_name": symbol, "expiry_type": "weekly"}
    if symbol in MONTHLY_INDEX_SYMBOLS:
        return {"asset_class": "index", "index_name": symbol, "expiry_type": "monthly"}
    if symbol in INDEX_SYMBOLS:
        return {"asset_class": "index", "index_name": symbol, "expiry_type": "monthly"}
    return {"asset_class": "equity", "index_name": None, "expiry_type": "monthly"}


def get_watchlist():
    items = []
    for entry in CONFIG.get("watchlist", []):
        if isinstance(entry, str):
            symbol = normalize_symbol(entry)
            profile = get_symbol_profile(symbol)
            items.append({
                "exchange": "NSE",
                "symbol": symbol,
                "type": profile["asset_class"],
                **profile,
            })
        else:
            symbol = normalize_symbol(entry["symbol"])
            profile = get_symbol_profile(symbol)
            items.append({
                "exchange": entry.get("exchange", "NSE"),
                "symbol": symbol,
                "type": profile["asset_class"],
                **profile,
            })
    return items


def get_active_watchlist():
    active = {normalize_symbol(s) for s in CONFIG.get("collection", {}).get("active_symbols", [])}
    if not active:
        return get_watchlist()
    return [item for item in get_watchlist() if item["symbol"] in active]


def select_symbol(driver, symbol, symbol_type):
    dropdown_xpath = XPATH_INDEX_DROPDOWN if symbol_type == "index" else XPATH_EQUITY_DROPDOWN
    dropdown = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, dropdown_xpath))
    )
    Select(dropdown).select_by_visible_text(symbol)
    print(f"  Selected {symbol_type}: {symbol}")
    time.sleep(3)


def _expiry_limit_for_symbol(symbol):
    cfg = CONFIG["scraping"]
    symbol = normalize_symbol(symbol)
    if symbol in WEEKLY_INDEX_SYMBOLS:
        return cfg.get("nifty_weekly_expiries", 3)
    if symbol in INDEX_SYMBOLS:
        return cfg.get("monthly_index_expiries", 2)
    return cfg.get("equity_expiries", 1)


def get_expiries_for_symbol(driver, symbol, symbol_type=None):
    dropdown = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, XPATH_EXPIRY_DROPDOWN))
    )
    select = Select(dropdown)
    expiries = [
        opt.text.strip()
        for opt in select.options
        if opt.text.strip() and opt.text.strip().lower() != "select"
    ]
    limit = _expiry_limit_for_symbol(symbol)
    expiries = expiries[:limit]
    print(f"  Using {len(expiries)} expiries for {symbol}: {expiries}")
    return expiries


def get_underlying_spot(driver):
    """Read underlying spot from option-chain page header."""
    import re
    body = driver.find_element(By.TAG_NAME, "body").text
    patterns = [
        r"Underlying Index\s*:\s*[\w\s]+\s+([\d,]+\.?\d*)",
        r"Underlying Value\s*:\s*([\d,]+\.?\d*)",
        r"Last Price\s*:\s*([\d,]+\.?\d*)",
    ]
    for pat in patterns:
        m = re.search(pat, body, re.IGNORECASE)
        if m:
            return float(m.group(1).replace(",", ""))
    return None


def get_strike_table_id(symbol_type):
    return "optionChainTable-indices" if symbol_type == "index" else "optionChainTable-equity"


def wait_for_option_chain_table(driver, symbol_type, timeout=20):
    table_id = get_strike_table_id(symbol_type)
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.ID, table_id))
    )
    # Wait for at least one strike row to populate
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located(
            (By.XPATH, f'//table[@id="{table_id}"]//tbody//tr//td[12]')
        )
    )


def find_download_csv_button(driver, timeout=20):
    """Locate NSE chain CSV download control (DOM id changed mid-2026)."""
    selectors = [
        (By.ID, "downloadOCTable"),
        (By.XPATH, XPATH_DOWNLOAD_CSV),
        (By.XPATH, '//a[@id="downloadOCTable"]//span[starts-with(@id,"download_csv")]'),
        (By.XPATH, '//a[@id="downloadOCTable"]//span[@id="download_csv"]'),
    ]
    deadline = time.time() + timeout
    while time.time() < deadline:
        for by, sel in selectors:
            try:
                el = driver.find_element(by, sel)
                if el.is_displayed():
                    return el
            except NoSuchElementException:
                continue
        time.sleep(0.5)
    raise TimeoutException(f"Download CSV button not found after {timeout}s")


def get_strike_map(driver, symbol_type):
    table_id = get_strike_table_id(symbol_type)
    strike_cells = driver.find_elements(
        By.XPATH, f'//table[@id="{table_id}"]//tbody//tr//td[12]'
    )
    strike_map = {}
    for i, cell in enumerate(strike_cells):
        text = cell.text.strip().replace(",", "")
        if not text:
            continue
        try:
            val = float(text)
            strike_map[i + 1] = str(int(val)) if val == int(val) else str(val)
        except ValueError:
            strike_map[i + 1] = text

    if strike_map:
        print(f"  Mapped {len(strike_map)} strikes from table")
    else:
        print("  [WARN] Could not read strikes from table, will extract from modal")
    return strike_map


import sys as _sys
if PROJECT_ROOT not in _sys.path:
    _sys.path.insert(0, PROJECT_ROOT)
from src.expiry_utils import enrich_chain_metadata


def save_chain_metadata(csv_path, meta, is_front=None):
    import json
    enrich_chain_metadata(meta, is_front=is_front)
    meta_path = csv_path.replace(".csv", ".meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    return meta_path


def canary_check_chain(csv_path, min_rows=10):
    import pandas as pd
    df = pd.read_csv(csv_path, skiprows=1)
    rows = len(df.dropna(how="all"))
    ok = rows >= min_rows
    status = "PASS" if ok else "FAIL"
    print(f"    Canary [{status}]: {rows} data rows in {os.path.basename(csv_path)}")
    return ok


print(f"Watchlist: {len(get_watchlist())} symbols | Active (Sprint 1): {len(get_active_watchlist())}")